# 03 · Elongation split into rate × length, and Bliss independence of each part

The elongation velocity of a mother is the product of two things,

$$v=\frac{dL}{dt}=\underbrace{\frac{d\ln L}{dt}}_{\lambda\ \text{(elongation rate, 1/min)}}\times\underbrace{L}_{\text{length}},$$

so a drug can change $v$ by changing how fast the cell makes new length per unit of length ($\lambda$), or by
changing how long the cell is ($L$, e.g. a filament that stops dividing but keeps growing). A faster velocity can
hide a slower rate on a much longer cell.

Given a **control** (e.g. LB), two **monotherapies** A and B and their **combination**, for a window of time that
you choose, the notebook:

1. splits the mean velocity of each condition **exactly** into rate × length × a coupling term;
2. gives the **effect of each drug on each part** (log-effects relative to the control, with 95 % intervals);
3. gives the **Bliss prediction** of the combination for each part, $\varepsilon_{\rm Bliss} = \varepsilon_A + \varepsilon_B$, and
   compares it with the observed combination and with each drug alone.

### How to use it

1. Fill in the cell **Your data** below: one line per folder of Excel files (`*_longitud_madres.xlsx`), with the
   areas you want and the condition they belong to.
2. Run all the cells (**Run › Run All Cells**).
3. The figures (PNG, 200 dpi, sized for slides), the tables (CSV) and the settings (`settings.json`) are saved in
   `results/03_growth_rate_length_bliss/<RUN_NAME>/`.

Nothing else has to be changed. The cells after **Your data** can be read but do not need to be edited.
Everything is done by functions of `mmtools` (the same ones a graphical interface would call:
`mm.run_growth_bliss(EXPERIMENTS, ...)` does the whole notebook in one line).

## Your data

**`EXPERIMENTS`**: one `dict` per folder and condition.

| key | what to write | example |
|---|---|---|
| `folder` | folder with the `<area>-<date>_longitud_madres.xlsx` files (absolute, or relative to this notebook) | `'../November/6/Stacks/Excel'` |
| `condition` | name of the condition; several lines can share one condition (e.g. two experiments of the same treatment) | `'NIT'` |
| `areas` | `'all'`, a range `'1-10'`, or a list `'1-5, 8, 12-14'` | `'11-20'` |
| `flow_start` | time the flow started (`'HH:MM'`), which becomes time 0. If you leave it out (or `None`), time 0 is the **first image of that folder** | `'10:17'` |
| `name` | *(optional)* label of the experiment; by default the date in the file names | `'06/11'` |

**Other settings**

- `ROLES`: which condition is the control, drug A, drug B and the combination (names as in `EXPERIMENTS`).
  Other conditions can be in `EXPERIMENTS`: they appear in the figures and tables, but not in the Bliss analysis.
- `WINDOW`: `(start, end)` in minutes since time 0: the window of the Bliss analysis.
- `T_END`: nothing after this time is used (e.g. a gap in the imaging); `None` = the latest time all the conditions have images for.
- `RUN_NAME`: name of the folder where this run's results are saved.
- `UM_PER_PX`: µm per pixel to give lengths and velocities in µm; `None` = pixels.
- `COVER`: a mother is used in a window if its intervals cover at least this fraction of it.
- `DT_RANGE`: intervals between consecutive images outside this range (min) are not used.
- `N_BOOT`: bootstrap replicates for the 95 % intervals.
- `CONDITION_COLORS`: *(optional)* your own colours, e.g. `{'LB': '#52514e', 'NIT': '#d9a21b'}`.

In [ ]:
EXPERIMENTS = [
    dict(folder='../Analisis_NIT_GEN/data_gen_nit_bliss/LB_16_10_excel', areas='all',   condition='LB',      flow_start='10:45'),
    dict(folder='../November/6/Stacks/Excel',                         areas='31-40', condition='GEN',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='11-30', condition='NIT',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='1-10',  condition='NIT+GEN', flow_start='10:17'),
]

ROLES = dict(control='LB', A='GEN', B='NIT', combination='NIT+GEN')
WINDOW = (125, 170)     # min since time 0
T_END = 170             # min, or None

RUN_NAME = 'NIT_GEN_125_170min'
UM_PER_PX = None        # e.g. 0.065; None = pixels
COVER = 0.6
DT_RANGE = (3, 25)      # min
N_BOOT = 1000
CONDITION_COLORS = {}   # optional, e.g. {'LB': '#52514e'}

## 1. Reading the Excel files

First the settings are checked (folders, areas, times); if something cannot be read, the notebook stops here and
says what. Then, for each folder, every `*_longitud_madres.xlsx` of the chosen areas is read: the time of each
image (row 1) and, for each channel, the mother's length and, in the row below, the daughter's length in the
image where a division is first seen.

In [ ]:
NOTEBOOK = '03_growth_rate_length_bliss'
SETTINGS = dict(roles=ROLES, window=list(WINDOW), t_end=T_END, um_per_px=UM_PER_PX, cover=COVER,
                dt_range=list(DT_RANGE), n_boot=N_BOOT)

import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import mmtools as mm
from mmtools import plots as mp
warnings.filterwarnings('ignore')
mp.set_style()

problems = mm.check_entries(EXPERIMENTS)
if problems:
    raise ValueError('Please fix the cell "Your data":\n  - ' + '\n  - '.join(problems))

ds = mm.load_dataset(EXPERIMENTS, step=1, t_end=T_END, condition_colors=CONDITION_COLORS)
print('\n'.join(ds.messages))
OUT = Path('results') / NOTEBOOK / RUN_NAME
res = mm.AnalysisResult(NOTEBOOK, SETTINGS, ds.experiments, messages=list(ds.messages))

def show(name, fig):
    # show a figure and keep it to be saved at the end
    res.figures[name] = fig
    plt.show()

ds.summary

## 2. Velocity = rate × length

Between two consecutive images of a mother, with $L_1$ the length in the first image and
$X_2 = L_2 + (\text{daughter released in between})$ the length it would have in the second if it had not divided:

| quantity | definition | meaning |
|---|---|---|
| velocity | $v = (X_2 - L_1)/\Delta t$ | new length per minute |
| rate | $\lambda = \ln(X_2/L_1)/\Delta t$ | new length per minute **per unit of length** (1/min); $\ln 2/\lambda$ is the doubling time |
| length | $\tilde L = (X_2 - L_1)/\ln(X_2/L_1)$ | the (logarithmic) mean length during the interval |

With these definitions $v = \lambda\,\tilde L$ **exactly** in every interval (it is the integral of $dL/dt = \lambda L$).

An interval is not used if one of the two lengths is missing (no reliable length, past the channel exit, mother
lost), if it contains a division whose daughter has no length, or if $\Delta t$ is outside `DT_RANGE`.

**One value per mother and window.** Each mother gets the time-weighted mean over its intervals inside the window
(an interval belongs to the window of its mid time): $v_m = \sum \Delta X/\sum\Delta t$,
$\lambda_m = \sum \ln(X_2/L_1)/\sum\Delta t$, $L_m = \sum \tilde L\,\Delta t/\sum\Delta t$. A mother counts if its
intervals cover at least `COVER` of the window. So everywhere in this notebook **one data point is one mother in
one window**, not one frame: the measurement noise of single intervals averages out, and the unit of the
distributions is the cell.

The table: number of mothers in the window, means and medians, doubling time $\ln 2/\langle\lambda\rangle$,
divisions per mother and hour, and the coupling $\kappa$ (section 5).

In [ ]:
G = mm.growth_split(ds, ROLES, WINDOW, cover=COVER, dt_range=DT_RANGE, um_per_px=UM_PER_PX, n_boot=N_BOOT)
print(f'window {WINDOW[0]}–{WINDOW[1]} min')
G.summary.round(4)

## 3. Along time

Consecutive 15-min windows ending at `T_END`. Each dot is one mother in one window, drawn at the centre of the
window and shifted sideways by condition; the line joins the medians over mothers (only windows with ≥ 30
mothers). The blue band is the selected `WINDOW`. Use this figure to choose the window: the Bliss analysis is
easiest to read where the effects have settled.

In [ ]:
show('time_course', G.fig_time_course())

## 4. Rate against length, one dot per mother

Horizontal axis: mean length of the mother in the window (log scale); vertical axis: its mean rate. The dashed
curves are lines of constant velocity, $\lambda = v/L$: every mother on the same curve elongates the same length
per minute. The control cloud is repeated in grey, and the crosses mark the means $(\langle L\rangle, \langle\lambda\rangle)$
of the panel (black) and of the control (grey).

Read the arrow from the grey cross to the black one: **sideways** is the length effect, **down or up** is the rate
effect, and crossing the dashed curves is the change of velocity.

In [ ]:
show('rate_vs_length', G.fig_rate_vs_length())

## 5. The coupling term

For each mother $v_m \approx \lambda_m L_m$, but the mean of a product is not the product of the means:

$$\langle v\rangle=\langle\lambda\rangle\langle L\rangle\,\kappa,\qquad
\kappa=1+\underbrace{r_{\lambda L}\,\mathrm{CV}_\lambda\,\mathrm{CV}_L}_{\text{between mothers}}
+\underbrace{\frac{\langle v-\lambda L\rangle}{\langle\lambda\rangle\langle L\rangle}}_{\text{within mothers}},$$

with $r_{\lambda L}$ the correlation between rate and length across mothers and CV the coefficients of variation.
The first part is the covariance between mothers; the second is the same thing inside each mother (over the window
its length grows while its rate fluctuates). $\kappa = \langle v\rangle/(\langle\lambda\rangle\langle L\rangle)$ by definition,
so the identity is exact. $\kappa \approx 1$ when rate and length are uncorrelated or their spreads are small.

In [ ]:
G.kappa.round(3)

## 6. Effect of each drug on each part, and Bliss

Taking the logarithm of the ratio to the control, the effect on the velocity splits **exactly** into three terms:

$$\underbrace{\ln\frac{\langle v\rangle_c}{\langle v\rangle_{\rm control}}}_{\varepsilon^{v}_c}
=\underbrace{\ln\frac{\langle\lambda\rangle_c}{\langle\lambda\rangle_{\rm control}}}_{\varepsilon^{\lambda}_c\ \text{(rate)}}
+\underbrace{\ln\frac{\langle L\rangle_c}{\langle L\rangle_{\rm control}}}_{\varepsilon^{L}_c\ \text{(length)}}
+\underbrace{\ln\frac{\kappa_c}{\kappa_{\rm control}}}_{\varepsilon^{\kappa}_c\ \text{(coupling)}} .$$

**Bliss independence** is additivity of log-effects, $\varepsilon_{\rm Bliss} = \varepsilon_A + \varepsilon_B$
(the combination multiplies the fractions left by each drug), and it can be asked of **each term separately**.
Because the split is a sum, the Bliss prediction of the velocity is the sum of the Bliss predictions of its parts.

How to read the table:

- $\varepsilon = 0$: no effect; $e^{\varepsilon}-1$ is the relative change ($\varepsilon = -0.2$ is −18 %, $+0.5$ is +65 %);
- **observed − Bliss** $< 0$: the combination lowers that part more than Bliss predicts (synergy for an inhibition),
  $> 0$: less (antagonism); compatible with 0: Bliss independence;
- **observed − A alone / B alone** $\approx 0$: the combination behaves like that drug alone for that part.

95 % intervals: bootstrap over **areas**, resampled with replacement within each condition (mothers of the same
area are not independent). With one experiment per condition they do not include day-to-day variation.

In [ ]:
print(f"log-effects relative to {ROLES['control']}, window {WINDOW[0]}–{WINDOW[1]} min (95 % bootstrap over areas)")
display(G.effects)
show('log_effects', G.fig_effects())

**Left:** the log-effect on the velocity (diamonds, with 95 % intervals) of each drug, the combination and the
Bliss prediction, split into rate (blue), length (orange) and coupling (grey) bars that add up to it.
**Right:** the combination against Bliss and against each drug alone, part by part.

**The Bliss prediction in the units of the data**, $\langle x\rangle_{\rm Bliss} = \langle x\rangle_{\rm control}\cdot
\frac{\langle x\rangle_A}{\langle x\rangle_{\rm control}}\cdot\frac{\langle x\rangle_B}{\langle x\rangle_{\rm control}}$,
against the observed combination:

In [ ]:
G.bliss.round(4)

## 7. The same split along time

The log-effects in 30-min windows moved every 15 min (the last one ends at `T_END`), drawn at the centre of each
window. Lines: each drug and the observed combination; dashed: Bliss. Bands: 95 % bootstrap over areas. The blue
band is the selected `WINDOW`. This shows whether the conclusion in the chosen window depends on the window.

In [ ]:
G.along_time(width=30, step=15, cover=COVER, n_boot=N_BOOT)
show('log_effects_along_time', G.fig_effects_time())

## 8. Saving

In `results/03_growth_rate_length_bliss/<RUN_NAME>/`: the figures (numbered PNG) and

- `window_summary.csv`, `coupling.csv`: sections 2 and 5;
- `log_effects.csv` (with intervals, as in the table), `log_effects_numbers.csv` (the same as numbers, with
  relative change in % and a two-sided bootstrap p-value), `bliss_prediction.csv`, `log_effects_along_time.csv`;
- `mothers_in_window.csv`: $v$, $\lambda$, $L$ and divisions per hour of every mother in the window;
- `experiments.csv`, `settings.json` (`mm.load_config` reads it back), `messages.txt`.

In [ ]:
res.tables.update(G.tables())
res.tables['experiments'] = ds.summary
res.save(OUT)
print('saved in', OUT.resolve())
sorted(p.name for p in OUT.iterdir())